# Python and Jupyter Notebooks · your first hour writing code

Every lab after this one is driven by Python running inside a Jupyter notebook, the same kind of notebook you are reading right now. Before you build edge systems, you need to be comfortable running cells, keeping track of what the notebook remembers, and reading and editing small pieces of Python.

This short pre-lab is the companion to the Linux Command Basics lab. That one taught you to drive the machine from the shell. This one teaches you to drive the notebook and read the Python the later labs are made of.

Work through it top to bottom. Run every code cell, read what comes back, and try the small exercises. Nothing here can harm the machine, so experiment freely.

## How this notebook works · cells and the kernel

- A **cell** is one box of code. Run it by clicking it and pressing **Shift+Enter**. The output appears right below it.
- Behind the notebook runs a **kernel**, a single Python session that stays alive as you go. Anything you define in one cell is remembered by every cell you run afterward.
- Cells run in the order **you** run them, not top to bottom on their own. If things get confusing, use **Kernel > Restart Kernel** and run the cells again from the top for a clean slate.

You saw `!` and `%%bash` cells in the Linux lab. Those run shell commands. In this lab most cells are plain Python, and Part 7 shows how the two work together.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

### Preflight · check your environment

In [ ]:
preflight([
    check("python 3 available", commandOnPath("python3"),
          hint="python3 runs every cell in this notebook. It ships with the DGX Spark."),
    check("os module importable", pythonImportable("os"),
          hint="os is part of Python's standard library and should always import."),
    check("pathlib module importable", pythonImportable("pathlib"),
          hint="pathlib is part of Python's standard library and should always import."),
    check("your home folder is writable", dirExists("~"),
          hint="You need a home folder to create the lab files. Ask your instructor if this fails."),
])

---
## Part 1 · Cells and the kernel

Run a cell and watch the kernel remember what you did.

**[Notebook cell]** Run your first cell. It does some arithmetic and prints the answer:

In [ ]:
print("2 + 2 =", 2 + 2)

**[Notebook cell]** The kernel remembers values between cells. Define one here:

In [ ]:
labName = "pythonBasicsLab"

**[Notebook cell]** The next cell can use it, even though you never repeat it. That is the kernel holding your session in memory:

In [ ]:
print("This lab is called", labName)

**[Notebook cell]** Make a folder to work in for the rest of the lab. You will write files into it and the checkpoints will look for them there. `Path.home()` is your home folder, and `mkdir` creates the folder:

In [ ]:
from pathlib import Path

labDir = Path.home() / labName
labDir.mkdir(exist_ok=True)
print("working in", labDir)

In [ ]:
checkpoint("Part 1 - cells and the kernel", [
    check("working folder exists", dirExists("~/pythonBasicsLab"),
          hint="Run the cell that creates labDir with mkdir."),
], successNote="You can run cells and the kernel remembers what you defined. That is the whole model.")

---
## Part 2 · Values, variables, and text

The three kinds of value you will see most: numbers, strings (text), and the f-string that stitches them together.

**[Notebook cell]** A number and a string. Notice text goes inside quotes and numbers do not:

In [ ]:
deviceCount = 3
siteName = "riverside"
print(deviceCount, siteName)

**[Notebook cell]** An **f-string** builds text with values dropped into it. Put an `f` before the quotes and wrap each value in `{ }`:

In [ ]:
message = f"Site {siteName} has {deviceCount} devices online"
print(message)

**[Notebook cell]** Save that message to a file so the checkpoint can find it. `write_text` writes a string to a file in one step:

In [ ]:
(labDir / "status.txt").write_text(message + "\n")
print("wrote", labDir / "status.txt")

**Try it:** change `siteName` in the cell above to your own name, then re-run that cell and the two cells after it. Watch the file change.

In [ ]:
checkpoint("Part 2 - values and text", [
    check("status.txt was written", fileNonEmpty("~/pythonBasicsLab/status.txt"),
          hint="Run the write_text cell above."),
    check("status.txt reports the devices", fileContains("~/pythonBasicsLab/status.txt", "devices online"),
          hint="Run the f-string cell, then the write_text cell."),
], successNote="Numbers, strings, and f-strings are most of the values you will touch in the later labs.")

---
## Part 3 · Lists and dictionaries

Two containers you will see in every lab. A **list** holds items in order. A **dictionary** holds labelled values, the way the lab setup cells pass settings like `{"MODEL": "yolov8n.pt"}`.

**[Notebook cell]** A list of sensor readings. Square brackets, comma separated. `len` counts them and `max` finds the biggest:

In [ ]:
readings = [21.5, 22.1, 20.8, 23.4]
print("count:", len(readings))
print("highest:", max(readings))

**[Notebook cell]** A dictionary maps a **key** to a **value**. Here the `id` is not made up: `platform.node()` asks the machine you are running on for its own hostname, and `os.cpu_count()` asks how many CPUs it has. Read a value back by putting its key in square brackets:

In [ ]:
import platform, os
device = {"id": platform.node(), "site": siteName, "cpus": os.cpu_count(), "online": True}
print(device["id"], "at", device["site"], "with", device["cpus"], "CPUs")

**[Notebook cell]** Save a one line summary built from both, then read it back with `read_text` to prove it landed:

In [ ]:
summary = f"{device['id']}: {device['cpus']} CPUs, {len(readings)} readings, peak {max(readings)}"
(labDir / "summary.txt").write_text(summary + "\n")
print((labDir / "summary.txt").read_text())

In [ ]:
import platform
checkpoint("Part 3 - lists and dictionaries", [
    check("summary.txt was written", fileNonEmpty("~/pythonBasicsLab/summary.txt"),
          hint="Run the cell that writes summary.txt."),
    check("summary names this machine", fileContains("~/pythonBasicsLab/summary.txt", platform.node()),
          hint="Run the dictionary cell, then the summary cell."),
], successNote="Lists and dictionaries carry almost all the data you will pass around in this course.")

---
## Part 4 · Decisions and loops

`if` chooses. `for` repeats. Together they are how a few lines of Python handle a whole list of readings.

**[Notebook cell]** `if` runs a block only when its test is true, and `else` covers the rest. Watch the indentation: Python uses it to group the block that belongs to each line:

In [ ]:
threshold = 22.0
for value in readings:
    if value > threshold:
        print(value, "is high")
    else:
        print(value, "is ok")

**[Notebook cell]** Now use a loop to write one line per reading into a log file. `enumerate` numbers the items, and `"\n".join(...)` joins the lines with line breaks:

In [ ]:
lines = []
for index, value in enumerate(readings, start=1):
    state = "HIGH" if value > threshold else "ok"
    lines.append(f"reading {index}: {value} {state}")

(labDir / "readings.log").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

In [ ]:
checkpoint("Part 4 - decisions and loops", [
    check("readings.log has one line per reading", fileNonEmpty("~/pythonBasicsLab/readings.log", minLines=4),
          hint="Run the loop cell that writes readings.log."),
    check("the log flags the high reading", fileContains("~/pythonBasicsLab/readings.log", "HIGH"),
          hint="One reading is above the threshold. Run the loop cell above."),
], successNote="if and for let a few lines of code handle a whole stream of data, which is exactly what the telemetry labs do.")

---
## Part 5 · Functions and modules

A **function** is a named block you can run again and again. `import` brings in code someone else wrote, like the `os` and `pathlib` modules every lab's setup cell uses.

**[Notebook cell]** Define a function with `def`, then call it by name. The value after `return` comes back to you:

In [ ]:
def label(value, limit):
    if value > limit:
        return "HIGH"
    return "ok"

print(label(25.0, threshold))
print(label(19.0, threshold))

**[Notebook cell]** `import` loads a module. `os` reads things about your session, like your username. This is the pattern you will see at the top of every later lab:

In [ ]:
import os

userName = os.environ.get("USER", "student")
print("you are", userName)

**[Notebook cell]** Put them together: tag each reading with your username using the function you just wrote:

In [ ]:
tagged = []
for value in readings:
    tagged.append(f"{userName}: {value} {label(value, threshold)}")

(labDir / "tagged.txt").write_text("\n".join(tagged) + "\n")
print("\n".join(tagged))

In [ ]:
checkpoint("Part 5 - functions and modules", [
    check("tagged.txt was written", fileNonEmpty("~/pythonBasicsLab/tagged.txt", minLines=4),
          hint="Run the cell that writes tagged.txt."),
    check("os imports cleanly", pythonImportable("os"),
          hint="Run the import cell. os is part of Python and should always load."),
], successNote="Functions and imports are the backbone of the toolkit cell at the top of every lab in this course.")

---
## Part 6 · Reading and writing files

You already wrote files with `write_text`. Here is the round trip: write, read back, and list what is in your lab folder.

**[Notebook cell]** Read the whole file back into a string and print it:

In [ ]:
text = (labDir / "readings.log").read_text()
print(text)

**[Notebook cell]** List every file you have created. `iterdir` walks the folder and `sorted` puts the names in order:

In [ ]:
for path in sorted(labDir.iterdir()):
    print(path.name)

**[Notebook cell]** Count the lines in the log and save a small report. `splitlines()` breaks the text into a list of lines so `len` can count them:

In [ ]:
lineCount = len(text.splitlines())
report = f"readings.log has {lineCount} lines"
(labDir / "report.txt").write_text(report + "\n")
print(report)

In [ ]:
checkpoint("Part 6 - reading and writing files", [
    check("report.txt was written", fileNonEmpty("~/pythonBasicsLab/report.txt"),
          hint="Run the cell that writes report.txt."),
    check("report counts the lines", fileContains("~/pythonBasicsLab/report.txt", "lines"),
          hint="Run the line counting cell above."),
], successNote="Read, transform, write. Almost every lab is some version of this loop over real device data.")

---
## Part 7 · When things break, and mixing shell with Python

Two survival skills: reading an error, and running shell commands from Python the way the Linux lab did.

**[Notebook cell]** Errors are normal. Run this one on purpose to see what an error looks like. Python prints a **traceback**, and the important part is the **last line**, which names the problem:

In [ ]:
print(totalCount)

**[Notebook cell]** The last line said `NameError: name 'totalCount' is not defined`, meaning you used a name before defining it. Define it first, then it works. When errors pile up and you lose track of what is defined, use **Kernel > Restart Kernel** and run again from the top:

In [ ]:
totalCount = len(readings)
print("total readings:", totalCount)

**[Notebook cell]** Finally, the bridge back to the Linux lab. A `!` cell runs a shell command, and you can capture its output into a Python list. Here `ls` lists your lab folder and Python counts the files:

In [ ]:
files = !ls ~/pythonBasicsLab
print("you created", len(files), "files:")
for name in files:
    print(" ", name)

In [ ]:
checkpoint("Part 7 - errors and the shell", [
    check("all lab files are present", fileExists("~/pythonBasicsLab/tagged.txt"),
          hint="Work through Parts 1 to 6 so every file gets written."),
    check("report is readable from the shell", outputContains("cat ~/pythonBasicsLab/report.txt", "lines"),
          hint="Run Part 6 so report.txt exists."),
], successNote="You can read an error and reach the shell from Python. That is enough to work through every lab that follows.")

---
**[Notebook cell]** Optional cleanup. Uncomment and run to remove the lab folder when you are done.

In [ ]:
# import shutil
# shutil.rmtree(Path.home() / "pythonBasicsLab", ignore_errors=True)
# print("cleaned up")

### Lab scorecard

In [ ]:
labSummary("Python and Jupyter Notebooks")

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Python and Jupyter Notebooks")

---

That is the whole toolkit: run cells, keep track of what the kernel remembers, and read and edit small pieces of Python. You will lean on all three in every lab that follows, starting the first time you build and run your own containers.